# Tercera Iteración: Arquitectura de Supervisor Neuronal (Deep Stacking MLP)

### Objetivo:
Usar todo el poder de una **Red Neuronal Artificial (MLP)** como meta-evaluador. La red analizará todas las características originales de los clientes JUNTO con las predicciones generadas por **LightGBM** y **CatBoost**, aprendiendo a corregir los errores que los modelos de árboles no pueden resolver por sí solos.

### Flujo de Datos (Prevención de Data Leakage):
1. **Partición Inicial (Holdout):** Separamos 10% de los datos como **Test Puro**. Estos datos NADIE los toca hasta el final (ni los árboles ni la red) para sacar la Matriz de Confusión.
2. **Entrenamiento Base (90%):** Usamos validación cruzada (K-Fold) sobre el 90% restante. Los árboles predicen sobre los folds que no vieron (OOF). Esto asegura que la predicción que le pasamos a la ARN sea un error honesto, evitando el Data Leakage.
3. **Entrenamiento ARN:** La ARN toma las columnas originales (escaladas) + OOF de los árboles. Internamente usa el 10% de este bloque para monitorear su propio AUC por época (Early Stopping).
4. **Evaluación Final:** Pasamos el 10% de Test Puro por los árboles, juntamos todo y la ARN da el veredicto final.

In [ ]:
import warnings
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, accuracy_score

warnings.filterwarnings('ignore')
print("✅ Librerías importadas.")

In [ ]:
# 1. CARGA Y PARTICIÓN DEL DATASET
train_df = pd.read_csv('../../dataset/train.csv')

# Separamos X e Y
X = train_df.drop(['id_cliente', 'objetivo'], axis=1)
y = train_df['objetivo']

# Extraemos el 10% para el Test Final (Matriz de Confusión)
X_train_full, X_test_final, y_train_full, y_test_final = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=42
)

print(f"Datos para Entrenamiento y Validación (90%): {X_train_full.shape[0]} filas")
print(f"Datos para Test Puro / Matriz de Confusión (10%): {X_test_final.shape[0]} filas")

In [ ]:
# 2. DEFINIR VARIABLES Y PREPROCESAMIENTO
cat_cols = ['ocupacion', 'region', 'canal_adquisicion', 'banda_riesgo', 'dispositivo_principal']
num_cols = [c for c in X.columns if c not in cat_cols]

X_train_cb = X_train_full.copy()
X_test_cb = X_test_final.copy()
for col in cat_cols:
    X_train_cb[col] = X_train_cb[col].astype(str)
    X_test_cb[col] = X_test_cb[col].astype(str)
    X_train_full[col] = X_train_full[col].astype('category')
    X_test_final[col] = X_test_final[col].astype('category')

In [ ]:
# 3. FASE BASE: ENTRENAMIENTO DE ÁRBOLES EXPERTOS (OOF para prevenir Data Leakage)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X_train_full))
test_preds_lgb = np.zeros(len(X_test_final))

oof_cb = np.zeros(len(X_train_full))
test_preds_cb = np.zeros(len(X_test_final))

best_lgb_params = {'num_leaves': 42, 'learning_rate': 0.017766463699737917, 'min_child_samples': 44, 'subsample': 0.6548629928035845, 'colsample_bytree': 0.595372408030665, 'reg_alpha': 4.9893861303564696e-05, 'reg_lambda': 0.1440126277634314}

print("🌳 Entrenando Expertos (LightGBM y CatBoost)...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):
    # LightGBM
    X_tr_lgb, y_tr_lgb = X_train_full.iloc[train_idx], y_train_full.iloc[train_idx]
    X_va_lgb, y_va_lgb = X_train_full.iloc[val_idx], y_train_full.iloc[val_idx]
    
    model_lgb = lgb.LGBMClassifier(objective='binary', metric='auc', n_estimators=1500, random_state=42, verbose=-1, **best_lgb_params)
    model_lgb.fit(X_tr_lgb, y_tr_lgb, eval_set=[(X_va_lgb, y_va_lgb)], callbacks=[lgb.early_stopping(30, verbose=False)])
    oof_lgb[val_idx] = model_lgb.predict_proba(X_va_lgb)[:, 1]
    test_preds_lgb += model_lgb.predict_proba(X_test_final)[:, 1] / skf.n_splits
    
    # CatBoost
    X_tr_cb, y_tr_cb = X_train_cb.iloc[train_idx], y_train_full.iloc[train_idx]
    X_va_cb, y_va_cb = X_train_cb.iloc[val_idx], y_train_full.iloc[val_idx]
    
    model_cb = CatBoostClassifier(eval_metric='AUC', random_seed=42, early_stopping_rounds=30, verbose=False, thread_count=-1)
    train_pool = Pool(X_tr_cb, y_tr_cb, cat_features=cat_cols)
    val_pool = Pool(X_va_cb, y_va_cb, cat_features=cat_cols)
    test_pool = Pool(X_test_cb, cat_features=cat_cols)
    
    model_cb.fit(train_pool, eval_set=val_pool, use_best_model=True)
    oof_cb[val_idx] = model_cb.predict_proba(val_pool)[:, 1]
    test_preds_cb += model_cb.predict_proba(test_pool)[:, 1] / skf.n_splits

print(f"✅ OOF LightGBM AUC: {roc_auc_score(y_train_full, oof_lgb):.5f}")
print(f"✅ OOF CatBoost AUC: {roc_auc_score(y_train_full, oof_cb):.5f}")

In [ ]:
# 4. PREPARACIÓN DEL SÚPER-DATASET PARA LA RED NEURONAL
print("⚙️ Transformando columnas originales y acoplando predicciones de los árboles...")
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# Ajustar escalador solo en datos de entrenamiento
X_train_rn_base = preprocessor.fit_transform(X_train_full)
X_test_rn_base = preprocessor.transform(X_test_final)

# Concatenamos [Features Originales] + [Opinion LGBM] + [Opinion CatBoost]
X_train_meta = np.column_stack([X_train_rn_base, oof_lgb, oof_cb])
X_test_meta = np.column_stack([X_test_rn_base, test_preds_lgb, test_preds_cb])

print(f"Dimensiones finales del Súper-Dataset: {X_train_meta.shape[1]} columnas (features + opiniones)")

In [ ]:
# 5. ENTRENAMIENTO DEL SUPERVISOR NEURONAL (MLP MEJORADO)
print("⚖️ Balanceando las clases para la Red Neuronal (Oversampling)...")
from sklearn.utils import resample

# Juntamos X e Y temporalmente para balancear
train_meta_df = pd.DataFrame(X_train_meta)
train_meta_df['target_y'] = y_train_full.values

df_majority = train_meta_df[train_meta_df['target_y'] == 0]
df_minority = train_meta_df[train_meta_df['target_y'] == 1]

# Upsample de la clase minoritaria para igualar a la mayoritaria
df_minority_upsampled = resample(df_minority, 
                                 replace=True,     
                                 n_samples=len(df_majority),    
                                 random_state=42)

df_upsampled = pd.concat([df_majority, df_minority_upsampled])

X_train_meta_bal = df_upsampled.drop('target_y', axis=1).values
y_train_full_bal = df_upsampled['target_y'].values

print("🧠 Entrenando Supervisor Neuronal (MLP Classifier Profundo)...")
arn_supervisor = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64), # Más profundidad y neuronas
    activation='relu',
    solver='adam',
    alpha=0.1,  # Mayor regularización L2 para evitar sobreajuste
    learning_rate_init=0.0005, # Aprendizaje más lento y seguro
    early_stopping=True,
    validation_fraction=0.1,
    max_iter=1000, # Aumentamos épocas máximas
    n_iter_no_change=30, # Aumentamos la paciencia del Early Stopping
    random_state=42,
    verbose=True
)

arn_supervisor.fit(X_train_meta_bal, y_train_full_bal)
print("\n✅ Supervisor Neuronal entrenado con éxito.")

In [ ]:
# 6. EVALUACIÓN FINAL SOBRE EL 10% TEST PURO
print("\n=======================================================")
print("📊 RESULTADOS FINALES SOBRE EL SET DE PRUEBA (10% PURO)")
print("=======================================================")

# La ARN emite su veredicto final
final_probs = arn_supervisor.predict_proba(X_test_meta)[:, 1]
final_preds_bin = arn_supervisor.predict(X_test_meta)

final_auc = roc_auc_score(y_test_final, final_probs)
final_gini = 2.0 * final_auc - 1.0
final_acc = accuracy_score(y_test_final, final_preds_bin)

print(f"⭐ AUC FINAL:  {final_auc:.5f}")
print(f"🏆 GINI FINAL: {final_gini:.5f}")
print(f"🎯 ACCURACY:   {final_acc:.5f}\n")

print("📉 MATRIZ DE CONFUSIÓN:")
print(confusion_matrix(y_test_final, final_preds_bin))
print("\n📋 REPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test_final, final_preds_bin))